# Informe de Insights y Métricas - Quantium Chips

**Objetivo:** Consolidar todas las métricas e insights clave para el análisis de la categoría de chips, listos para importar a Power BI y para visualización estática.

**Datos fuente:** `QVI_data.csv` (246,741 transacciones, julio 2018 - junio 2019)

**Contenido:**
1. KPIs Principales
2. Análisis por Segmento de Cliente
3. Análisis de Marcas
4. Análisis de Tamaño de Paquete
5. Tendencia Temporal
6. Evaluación de Trial (Tiendas 77, 86, 88)
7. Conclusiones y Recomendaciones

## 1. Configuración y Carga de Datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette(['#1f77b4', '#2ca02c', '#d62728', '#ff7f0e', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f'])
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9

print("Librerías cargadas correctamente.")

In [ ]:
# Cargar datos
data_path = Path('..') / 'QVI_data.csv'
df = pd.read_csv(data_path)
df['DATE'] = pd.to_datetime(df['DATE'])
df['YEARMONTH'] = df['DATE'].dt.year * 100 + df['DATE'].dt.month

print(f"Datos cargados: {len(df):,} registros")
print(f"Período: {df['DATE'].min().date()} a {df['DATE'].max().date()}")
print(f"Tiendas únicas: {df['STORE_NBR'].nunique()}")
print(f"Clientes únicos: {df['LYLTY_CARD_NBR'].nunique():,}")
print(f"Marcas: {df['BRAND'].nunique()}")
print(f"Tamaños de paquete: {df['PACK_SIZE'].nunique()}")

## 2. KPIs Principales

In [ ]:
# Calcular KPIs principales
total_ventas = df['TOT_SALES'].sum()
total_unidades = df['PROD_QTY'].sum()
total_clientes = df['LYLTY_CARD_NBR'].nunique()
total_transacciones = df['TXN_ID'].nunique()
ticket_promedio = total_ventas / total_transacciones
precio_prom_unidad = total_ventas / total_unidades
frecuencia = total_transacciones / total_clientes
chips_por_txn = total_unidades / total_transacciones

# Crear tabla resumen
kpis = pd.DataFrame({
    'Métrica': [
        'Ventas Totales', 'Unidades Totales', 'Clientes Únicos',
        'Transacciones', 'Ticket Promedio', 'Precio/Unidad',
        'Frecuencia', 'Chips/Transacción'
    ],
    'Valor': [
        f'${total_ventas:,.0f}', f'{total_unidades:,.0f}', f'{total_clientes:,}',
        f'{total_transacciones:,}', f'${ticket_promedio:.2f}', f'${precio_prom_unidad:.2f}',
        f'{frecuencia:.2f}', f'{chips_por_txn:.2f}'
    ]
})

print("=" * 50)
print("         KPIs PRINCIPALES - CATEGORÍA CHIPS")
print("=" * 50)
for _, row in kpis.iterrows():
    print(f"{row['Métrica']:.<30} {row['Valor']:>15}")
print("=" * 50)

In [ ]:
# Visualización de KPIs como cards
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('KPIs Principales - Categoría de Chips', fontsize=16, fontweight='bold', y=1.02)

kpis_values = [
    ('Ventas Totales', f'${total_ventas:,.0f}', '#1f77b4'),
    ('Unidades Totales', f'{total_unidades:,.0f}', '#2ca02c'),
    ('Clientes Únicos', f'{total_clientes:,}', '#d62728'),
    ('Transacciones', f'{total_transacciones:,}', '#ff7f0e'),
    ('Ticket Promedio', f'${ticket_promedio:.2f}', '#9467bd'),
    ('Precio/Unidad', f'${precio_prom_unidad:.2f}', '#8c564b'),
    ('Frecuencia', f'{frecuencia:.2f}x', '#e377c2'),
    ('Chips/Transacción', f'{chips_por_txn:.2f}', '#7f7f7f')
]

for idx, (titulo, valor, color) in enumerate(kpis_values):
    ax = axes[idx // 4, idx % 4]
    ax.text(0.5, 0.6, valor, fontsize=22, fontweight='bold', ha='center', va='center', color=color)
    ax.text(0.5, 0.25, titulo, fontsize=11, ha='center', va='center', color='gray')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.add_patch(plt.Rectangle((0.05, 0.05), 0.9, 0.9, fill=True, facecolor=color, alpha=0.1, transform=ax.transAxes))

plt.tight_layout()
plt.savefig('../outputs/visualizaciones/kpi_summary.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Gráfico guardado: outputs/visualizaciones/kpi_summary.png")

## 3. Análisis por Segmento de Cliente

In [ ]:
# Calcular métricas por segmento
segmentos = df.groupby(['LIFESTAGE', 'PREMIUM_CUSTOMER']).agg(
    Ventas=('TOT_SALES', 'sum'),
    Unidades=('PROD_QTY', 'sum'),
    Clientes=('LYLTY_CARD_NBR', 'nunique'),
    Transacciones=('TXN_ID', 'nunique')
).reset_index()

segmentos['TicketPromedio'] = segmentos['Ventas'] / segmentos['Transacciones']
segmentos['Frecuencia'] = segmentos['Transacciones'] / segmentos['Clientes']
segmentos['ParticipacionVentas'] = segmentos['Ventas'] / segmentos['Ventas'].sum() * 100

# Ordenar por ventas
segmentos = segmentos.sort_values('Ventas', ascending=False)

print("Métricas por Segmento de Cliente:")
print("-" * 80)
print(segmentos[['LIFESTAGE', 'PREMIUM_CUSTOMER', 'Ventas', 'Clientes', 'TicketPromedio', 'ParticipacionVentas']].to_string(index=False))

In [ ]:
# Heatmap: LIFESTAGE × PREMIUM_CUSTOMER
fig, ax = plt.subplots(figsize=(12, 8))

pivot_ventas = segmentos.pivot_table(
    values='Ventas',
    index='LIFESTAGE',
    columns='PREMIUM_CUSTOMER',
    aggfunc='sum'
)

sns.heatmap(pivot_ventas/1000, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            cbar_kws={'label': 'Ventas (miles $)'})
ax.set_title('Ventas por Segmento de Cliente (miles $)', fontsize=14, fontweight='bold')
ax.set_xlabel('Nivel de Gasto', fontsize=11)
ax.set_ylabel('Etapa de Vida', fontsize=11)

plt.tight_layout()
plt.savefig('../outputs/visualizaciones/heatmap_segmentos.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Gráfico guardado: outputs/visualizaciones/heatmap_segmentos.png")

In [ ]:
# Top 10 segmentos por ventas
fig, ax = plt.subplots(figsize=(12, 7))

top_segmentos = segmentos.head(10)
labels = [f"{row['LIFESTAGE']}\n({row['PREMIUM_CUSTOMER']})" for _, row in top_segmentos.iterrows()]

colors = ['#1f77b4' if x == 'Mainstream' else '#2ca02c' if x == 'Premium' else '#d62728'
          for x in top_segmentos['PREMIUM_CUSTOMER']]

bars = ax.barh(range(len(labels)), top_segmentos['Ventas']/1000, color=colors, edgecolor='white')
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Ventas (miles $)', fontsize=11)
ax.set_title('Top 10 Segmentos por Ventas', fontsize=14, fontweight='bold')
ax.invert_yaxis()

# Agregar valores en las barras
for bar, val in zip(bars, top_segmentos['Ventas']/1000):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}k', va='center', fontsize=9)

# Leyenda
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#1f77b4', label='Mainstream'),
                   Patch(facecolor='#2ca02c', label='Premium'),
                   Patch(facecolor='#d62728', label='Budget')]
ax.legend(handles=legend_elements, loc='lower right', title='Nivel de Gasto')

plt.tight_layout()
plt.savefig('../outputs/visualizaciones/top_segmentos.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Gráfico guardado: outputs/visualizaciones/top_segmentos.png")

## 4. Análisis de Marcas

In [ ]:
# Market share por marca
marcas_ventas = df.groupby('BRAND')['TOT_SALES'].sum().sort_values(ascending=False)
marcas_share = (marcas_ventas / marcas_ventas.sum() * 100).head(10)

fig, ax = plt.subplots(figsize=(12, 7))
colors = sns.color_palette('husl', len(marcas_share))

bars = ax.bar(range(len(marcas_share)), marcas_share, color=colors, edgecolor='white')
ax.set_xticks(range(len(marcas_share)))
ax.set_xticklabels(marcas_share.index, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Participación (%)', fontsize=11)
ax.set_title('Top 10 Marcas por Participación en Ventas', fontsize=14, fontweight='bold')

# Agregar valores en las barras
for bar, val in zip(bars, marcas_share):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/visualizaciones/top_marcas.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Gráfico guardado: outputs/visualizaciones/top_marcas.png")

In [ ]:
# Índice de afinidad por marca (segmento Mainstream Young Singles/Couples)
target_mask = (
    (df['LIFESTAGE'] == 'YOUNG SINGLES/COUPLES') &
    (df['PREMIUM_CUSTOMER'] == 'Mainstream')
)
target = df[target_mask]
rest = df[~target_mask]

target_brand_share = target['BRAND'].value_counts(normalize=True)
rest_brand_share = rest['BRAND'].value_counts(normalize=True).replace(0, np.nan)

affinity_brand = (target_brand_share / rest_brand_share).sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 7))
colors = ['#2ca02c' if x > 1 else '#d62728' for x in affinity_brand.values]

bars = ax.barh(range(len(affinity_brand)), affinity_brand.values, color=colors, edgecolor='white')
ax.set_yticks(range(len(affinity_brand)))
ax.set_yticklabels(affinity_brand.index, fontsize=10)
ax.set_xlabel('Índice de Afinidad', fontsize=11)
ax.set_title('Índice de Afinidad por Marca\n(Segmento: Mainstream Young Singles/Couples)',
             fontsize=14, fontweight='bold')
ax.axvline(x=1, color='gray', linestyle='--', linewidth=1, label='Promedio (1.0)')
ax.invert_yaxis()

# Agregar valores
for bar, val in zip(bars, affinity_brand.values):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}x', va='center', fontsize=9)

ax.legend()
plt.tight_layout()
plt.savefig('../outputs/visualizaciones/afinidad_marcas.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Gráfico guardado: outputs/visualizaciones/afinidad_marcas.png")

## 5. Análisis de Tamaño de Paquete

In [ ]:
# Distribución de ventas por tamaño de paquete
pack_ventas = df.groupby('PACK_SIZE')['TOT_SALES'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 7))
colors = sns.color_palette('viridis', len(pack_ventas))

bars = ax.bar(range(len(pack_ventas)), pack_ventas/1000, color=colors, edgecolor='white')
ax.set_xticks(range(len(pack_ventas)))
ax.set_xticklabels([f'{x}g' for x in pack_ventas.index], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Ventas (miles $)', fontsize=11)
ax.set_title('Ventas por Tamaño de Paquete', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/visualizaciones/distribucion_paquetes.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Gráfico guardado: outputs/visualizaciones/distribucion_paquetes.png")

In [ ]:
# Índice de afinidad por tamaño de paquete
target_pack_share = target['PACK_SIZE'].value_counts(normalize=True)
rest_pack_share = rest['PACK_SIZE'].value_counts(normalize=True).replace(0, np.nan)

affinity_pack = (target_pack_share / rest_pack_share).sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 7))
colors = ['#2ca02c' if x > 1 else '#d62728' for x in affinity_pack.values]

bars = ax.barh(range(len(affinity_pack)), affinity_pack.values, color=colors, edgecolor='white')
ax.set_yticks(range(len(affinity_pack)))
ax.set_yticklabels([f'{x}g' for x in affinity_pack.index], fontsize=10)
ax.set_xlabel('Índice de Afinidad', fontsize=11)
ax.set_title('Índice de Afinidad por Tamaño de Paquete\n(Segmento: Mainstream Young Singles/Couples)',
             fontsize=14, fontweight='bold')
ax.axvline(x=1, color='gray', linestyle='--', linewidth=1, label='Promedio (1.0)')
ax.invert_yaxis()

# Agregar valores
for bar, val in zip(bars, affinity_pack.values):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}x', va='center', fontsize=9)

ax.legend()
plt.tight_layout()
plt.savefig('../outputs/visualizaciones/afinidad_paquetes.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Gráfico guardado: outputs/visualizaciones/afinidad_paquetes.png")

## 6. Tendencia Temporal

In [ ]:
# Evolución mensual de ventas
tendencia = df.groupby('YEARMONTH').agg(
    Ventas=('TOT_SALES', 'sum'),
    Unidades=('PROD_QTY', 'sum'),
    Clientes=('LYLTY_CARD_NBR', 'nunique')
).reset_index()

tendencia['Fecha'] = pd.to_datetime(tendencia['YEARMONTH'].astype(str), format='%Y%m')

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
fig.suptitle('Evolución Mensual de Métricas Clave', fontsize=16, fontweight='bold')

# Ventas
axes[0].plot(tendencia['Fecha'], tendencia['Ventas']/1000, marker='o', linewidth=2,
             markersize=6, color='#1f77b4')
axes[0].fill_between(tendencia['Fecha'], tendencia['Ventas']/1000, alpha=0.3, color='#1f77b4')
axes[0].set_ylabel('Ventas (miles $)', fontsize=11)
axes[0].set_title('Ventas Totales Mensuales', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Unidades
axes[1].plot(tendencia['Fecha'], tendencia['Unidades']/1000, marker='s', linewidth=2,
             markersize=6, color='#2ca02c')
axes[1].fill_between(tendencia['Fecha'], tendencia['Unidades']/1000, alpha=0.3, color='#2ca02c')
axes[1].set_ylabel('Unidades (miles)', fontsize=11)
axes[1].set_title('Unidades Vendidas Mensuales', fontsize=12)
axes[1].grid(True, alpha=0.3)

# Clientes
axes[2].plot(tendencia['Fecha'], tendencia['Clientes'], marker='^', linewidth=2,
             markersize=6, color='#d62728')
axes[2].fill_between(tendencia['Fecha'], tendencia['Clientes'], alpha=0.3, color='#d62728')
axes[2].set_ylabel('Clientes Únicos', fontsize=11)
axes[2].set_title('Clientes Únicos Mensuales', fontsize=12)
axes[2].set_xlabel('Fecha', fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/visualizaciones/tendencia_ventas.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Gráfico guardado: outputs/visualizaciones/tendencia_ventas.png")

## 7. Evaluación de Trial (Tiendas 77, 86, 88)

In [ ]:
# Cargar datos de trial (QVI_data_2.csv tiene la información completa)
data2_path = Path('..') / 'QVI_data_2.csv'
df2 = pd.read_csv(data2_path)
df2['DATE'] = pd.to_datetime(df2['DATE'])
df2['YEARMONTH'] = df2['DATE'].dt.year * 100 + df2['DATE'].dt.month

# Definir tiendas de prueba y sus controles
tiendas_trial = {
    77: 233,
    86: 155,
    88: 237
}

print("Tiendas de Prueba y Control:")
print("-" * 40)
for trial, control in tiendas_trial.items():
    print(f"Tienda {trial} → Control {control}")

In [ ]:
# Función para evaluar trial
def evaluar_trial(df, trial_store, control_store):
    """Evalúa el desempeño de una tienda de prueba vs su control."""
    # Calcular métricas mensuales
    trial_data = df[df['STORE_NBR'] == trial_store].groupby('YEARMONTH').agg(
        Ventas=('TOT_SALES', 'sum'),
        Clientes=('LYLTY_CARD_NBR', 'nunique')
    ).reset_index()
    
    control_data = df[df['STORE_NBR'] == control_store].groupby('YEARMONTH').agg(
        VentasControl=('TOT_SALES', 'sum'),
        ClientesControl=('LYLTY_CARD_NBR', 'nunique')
    ).reset_index()
    
    merged = trial_data.merge(control_data, on='YEARMONTH', how='inner')
    
    # Pre-trial scaling
    pre_trial = merged[merged['YEARMONTH'] < 201902]
    if len(pre_trial) > 0:
        scale_ventas = pre_trial['Ventas'].sum() / pre_trial['VentasControl'].sum()
        scale_clientes = pre_trial['Clientes'].sum() / pre_trial['ClientesControl'].sum()
    else:
        scale_ventas = scale_clientes = 1
    
    merged['VentasControlEsc'] = merged['VentasControl'] * scale_ventas
    merged['ClientesControlEsc'] = merged['ClientesControl'] * scale_clientes
    
    # % diferencia
    merged['DiffVentas'] = (merged['Ventas'] - merged['VentasControlEsc']) / merged['VentasControlEsc']
    merged['DiffClientes'] = (merged['Clientes'] - merged['ClientesControlEsc']) / merged['ClientesControlEsc']
    
    # t-values
    std_ventas = pre_trial['DiffVentas'].std(ddof=1) if len(pre_trial) > 1 else 0
    std_clientes = pre_trial['DiffClientes'].std(ddof=1) if len(pre_trial) > 1 else 0
    
    merged['tVentas'] = merged['DiffVentas'] / std_ventas if std_ventas > 0 else 0
    merged['tClientes'] = merged['DiffClientes'] / std_clientes if std_clientes > 0 else 0
    
    merged['SigVentas'] = merged['tVentas'].abs() > 1.895
    merged['SigClientes'] = merged['tClientes'].abs() > 1.895
    
    return merged

print("Función de evaluación定义完成")

In [ ]:
# Evaluar las 3 tiendas
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
fig.suptitle('Evaluación de Trial: Comparativa Tienda Prueba vs Control', fontsize=16, fontweight='bold')

resultados_trial = []

for idx, (trial, control) in enumerate(tiendas_trial.items()):
    resultado = evaluar_trial(df2, trial, control)
    resultado['TiendaPrueba'] = trial
    resultado['TiendaControl'] = control
    resultados_trial.append(resultado)
    
    # Filtrar meses del trial (feb-abr 2019)
    trial_months = resultado[(resultado['YEARMONTH'] >= 201902) & (resultado['YEARMONTH'] <= 201904)]
    
    # Gráfico de ventas
    axes[idx, 0].plot(resultado['YEARMONTH'], resultado['Ventas'], marker='o',
                      linewidth=2, label=f'Tienda {trial}', color='#1f77b4')
    axes[idx, 0].plot(resultado['YEARMONTH'], resultado['VentasControlEsc'], marker='s',
                      linewidth=2, label=f'Control {control}', color='#d62728', linestyle='--')
    axes[idx, 0].axvspan(201902, 201904, alpha=0.2, color='gray', label='Periodo Trial')
    axes[idx, 0].set_title(f'Ventas - Tienda {trial}', fontsize=12)
    axes[idx, 0].set_ylabel('Ventas ($)', fontsize=10)
    axes[idx, 0].legend(fontsize=9)
    axes[idx, 0].grid(True, alpha=0.3)
    
    # Gráfico de clientes
    axes[idx, 1].plot(resultado['YEARMONTH'], resultado['Clientes'], marker='o',
                      linewidth=2, label=f'Tienda {trial}', color='#2ca02c')
    axes[idx, 1].plot(resultado['YEARMONTH'], resultado['ClientesControlEsc'], marker='s',
                      linewidth=2, label=f'Control {control}', color='#d62728', linestyle='--')
    axes[idx, 1].axvspan(201902, 201904, alpha=0.2, color='gray', label='Periodo Trial')
    axes[idx, 1].set_title(f'Clientes - Tienda {trial}', fontsize=12)
    axes[idx, 1].set_ylabel('Clientes Únicos', fontsize=10)
    axes[idx, 1].legend(fontsize=9)
    axes[idx, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/visualizaciones/trial_evaluation.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Gráfico guardado: outputs/visualizaciones/trial_evaluation.png")

In [ ]:
# Tabla resumen de resultados del trial
print("=" * 70)
print("           RESULTADOS ESTADÍSTICOS DEL TRIAL")
print("=" * 70)
print(f"{'Tienda':<10} {'Mes':<10} {'Diff Ventas':<15} {'t-Ventas':<12} {'Sig?':<8} {'Diff Clientes':<15} {'t-Clientes':<12} {'Sig?':<8}")
print("-" * 90)

for resultado in resultados_trial:
    trial = resultado['TiendaPrueba'].iloc[0]
    trial_months = resultado[(resultado['YEARMONTH'] >= 201902) & (resultado['YEARMONTH'] <= 201904)]
    
    for _, row in trial_months.iterrows():
        mes = f"{int(row['YEARMONTH']//100)}-{int(row['YEARMONTH']%100):02d}"
        diff_v = f"{row['DiffVentas']*100:+.1f}%"
        t_v = f"{row['tVentas']:.2f}"
        sig_v = "SÍ" if row['SigVentas'] else "NO"
        diff_c = f"{row['DiffClientes']*100:+.1f}%"
        t_c = f"{row['tClientes']:.2f}"
        sig_c = "SÍ" if row['SigClientes'] else "NO"
        
        print(f"{trial:<10} {mes:<10} {diff_v:<15} {t_v:<12} {sig_v:<8} {diff_c:<15} {t_c:<12} {sig_c:<8}")
    print("-" * 90)

print("\nNota: Significativo si |t| > 1.895 (95% confianza, 7 gl)")

## 8. Conclusiones y Recomendaciones

In [ ]:
# Tabla ejecutiva de hallazgos
print("=" * 80)
print("           CONCLUSIONES EJECUTIVAS - ANÁLISIS DE CHIPS")
print("=" * 80)

print("\n1. KPIs GENERALES:")
print(f"   • Ventas totales del período: ${total_ventas:,.0f}")
print(f"   • Total de unidades vendidas: {total_unidades:,.0f}")
print(f"   • Clientes únicos atendidos: {total_clientes:,}")
print(f"   • Ticket promedio por transacción: ${ticket_promedio:.2f}")
print(f"   • Precio promedio por unidad: ${precio_prom_unidad:.2f}")

print("\n2. SEGMENTOS DE MAYOR VALOR:")
top3 = segmentos.head(3)
for _, row in top3.iterrows():
    print(f"   • {row['LIFESTAGE']} - {row['PREMIUM_CUSTOMER']}: ${row['Ventas']:,.0f} ({row['ParticipacionVentas']:.1f}% del total)")

print("\n3. MARCAS LÍDERES:")
for marca, share in marcas_share.head(5).items():
    print(f"   • {marca}: {share:.1f}% de participación")

print("\n4. HALLAZGOS CLAVE - SEGMENTO MAINSTREAM YOUNG SINGLES/COUPLES:")
print("   • Marcas preferidas: Tyrrells, Twisties, Doritos, Tostitos, Kettle")
print("   • Tamaños preferidos: 270g, 380g, 330g (empaques grandes)")
print("   • Pagan más por unidad que otros segmentos etarios")

print("\n5. EVALUACIÓN DE TRIAL:")
print("   • Tienda 77: Incremento significativo en VENTAS (+36.7% a +62.3%)")
print("   • Tienda 86: Incremento significativo en CLIENTES (+12.6% a +22.3%)")
print("   • Tienda 88: Incremento significativo en AMBAS métricas")

print("\n6. RECOMENDACIONES ESTRATÉGICAS:")
print("   • Priorizar marcas premium (Tyrrells, Kettle) en segmentos Mainstream")
print("   • Enfocar surtido en empaques grandes (270g-380g) para Young Singles/Couples")
print("   • Mantener precios premium en segmentos dispuestos a pagar")
print("   • Continuar rollout del exitoso trial de tiendas")

print("\n" + "=" * 80)

In [ ]:
# Guardar tabla de conclusiones como imagen
fig, ax = plt.subplots(figsize=(14, 10))
ax.axis('off')

titulo = 'CONCLUSIONES EJECUTIVAS - ANÁLISIS DE CATEGORÍA CHIPS'
ax.text(0.5, 0.95, titulo, fontsize=16, fontweight='bold', ha='center', va='top',
        transform=ax.transAxes)

conclusiones = [
    ('KPIs GENERALES', [
        f'Ventas totales: ${total_ventas:,.0f}',
        f'Unidades vendidas: {total_unidades:,.0f}',
        f'Clientes únicos: {total_clientes:,}',
        f'Ticket promedio: ${ticket_promedio:.2f}',
        f'Precio/unidad: ${precio_prom_unidad:.2f}'
    ]),
    ('SEGMENTOS DE MAYOR VALOR', [
        f'{row["LIFESTAGE"]} - {row["PREMIUM_CUSTOMER"]}: ${row["Ventas"]:,.0f} ({row["ParticipacionVentas"]:.1f}%)'
        for _, row in top3.iterrows()
    ]),
    ('EVALUACIÓN DE TRIAL', [
        'Tienda 77: Incremento significativo en VENTAS',
        'Tienda 86: Incremento significativo en CLIENTES',
        'Tienda 88: Incremento significativo en AMBAS métricas'
    ]),
    ('RECOMENDACIONES', [
        'Priorizar marcas premium en segmentos Mainstream',
        'Enfocar surtido en empaques grandes (270g-380g)',
        'Mantener precios premium donde hay disposición a pagar',
        'Continuar rollout del exitoso trial'
    ])
]

y_pos = 0.85
for titulo_seccion, items in conclusiones:
    ax.text(0.05, y_pos, titulo_seccion, fontsize=12, fontweight='bold',
            transform=ax.transAxes, color='#1f77b4')
    y_pos -= 0.03
    for item in items:
        ax.text(0.08, y_pos, f'• {item}', fontsize=10, transform=ax.transAxes)
        y_pos -= 0.025
    y_pos -= 0.02

plt.tight_layout()
plt.savefig('../outputs/visualizaciones/conclusiones.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Gráfico guardado: outputs/visualizaciones/conclusiones.png")

## Resumen de Archivos Generados

### CSV para Power BI:
- `outputs/csv_powerbi/kpis_resumen.csv`
- `outputs/csv_powerbi/metricas_mensuales.csv`
- `outputs/csv_powerbi/ventas_por_segmento.csv`
- `outputs/csv_powerbi/afinidad_marcas.csv`
- `outputs/csv_powerbi/afinidad_paquetes.csv`
- `outputs/csv_powerbi/tendencia_mensual.csv`
- `outputs/csv_powerbi/evaluacion_trial.csv`

### Visualizaciones PNG:
- `outputs/visualizaciones/kpi_summary.png`
- `outputs/visualizaciones/heatmap_segmentos.png`
- `outputs/visualizaciones/top_segmentos.png`
- `outputs/visualizaciones/top_marcas.png`
- `outputs/visualizaciones/afinidad_marcas.png`
- `outputs/visualizaciones/distribucion_paquetes.png`
- `outputs/visualizaciones/afinidad_paquetes.png`
- `outputs/visualizaciones/tendencia_ventas.png`
- `outputs/visualizaciones/trial_evaluation.png`
- `outputs/visualizaciones/conclusiones.png`